# Practice & HomeTask: Conjugate Distributions
### Beta-Binomial & Gamma-Poisson Conjugate Models
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/04_conjugates_distributions_practice.ipynb)

---

## 1. Overview & Setup
This laboratory workbook guides you through closed-form conjugate Bayesian inference for:
1. **Beta–Binomial Model**: Modeling exam success proportions under partial knowledge vs. guessing.
2. **Beta–Bernoulli Model with Survey Data**: Estimating demographic and health behavior proportions using the CDC Behavioral Risk Factor Surveillance System (BRFSS).
3. **Gamma–Poisson Model**: Estimating daily dietary intake event rates (fruit & vegetable consumption) and computing 95% credible intervals.


In [ ]:
# ==============================================================================
# 🚀 Environment Setup & Self-Healing Data Fetch
# ==============================================================================
import os
import sys
import urllib.request
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

IS_COLAB = "google.colab" in sys.modules

# Check possible local paths
candidate_paths = [
    "data/brfss.csv",
    "../data/brfss.csv",
    "../../data/brfss.csv",
    "brfss.csv"
]
LOCAL_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        LOCAL_PATH = p
        break

if LOCAL_PATH is None or IS_COLAB:
    DATA_DIR = "data"
    os.makedirs(DATA_DIR, exist_ok=True)
    RAW_URL = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/brfss.csv"
    TARGET_PATH = os.path.join(DATA_DIR, "brfss.csv")
    if not os.path.exists(TARGET_PATH):
        print(f"📥 Downloading BRFSS dataset from {RAW_URL}...")
        urllib.request.urlretrieve(RAW_URL, TARGET_PATH)
        print("✅ Download complete.")
    LOCAL_PATH = TARGET_PATH

sns.set_style('whitegrid')
print(f"📂 Dataset ready at: {LOCAL_PATH}")


---
## 2. Foundations of Conjugate Updating

A prior $p(\theta)$ is **conjugate** to a likelihood $p(x \mid \theta)$ if the resulting posterior $p(\theta \mid x)$ belongs to the exact same parametric family as the prior.

### A. Beta–Binomial Model
* **Parameter**: Success probability $\theta \in [0, 1]$.
* **Prior**: $\theta \sim \operatorname{Beta}(\alpha, \beta)$
  $$p(\theta) = \frac{1}{\mathrm{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$
* **Likelihood**: $k$ successes in $n$ trials: $k \sim \operatorname{Binomial}(n, \theta)$
* **Posterior**:
  $$\theta \mid k \sim \operatorname{Beta}(\alpha + k, \; \beta + n - k)$$
* **Posterior Mean**:
  $$\mathbb{E}[\theta \mid k] = \frac{\alpha + k}{\alpha + \beta + n}$$

### B. Gamma–Poisson Model
* **Parameter**: Event rate intensity $\lambda > 0$.
* **Prior**: $\lambda \sim \operatorname{Gamma}(\alpha, \beta)$ (shape $\alpha$, rate $\beta$)
  $$p(\lambda) = \frac{\beta^\alpha}{\Gamma(\alpha)} \lambda^{\alpha - 1} e^{-\beta \lambda}$$
* **Likelihood**: $n$ independent count observations $y_1, \dots, y_n \sim \operatorname{Poisson}(\lambda)$:
  $$p(y \mid \lambda) = \prod_{i=1}^n \frac{\lambda^{y_i} e^{-\lambda}}{y_i!} \propto \lambda^{\sum y_i} e^{-n \lambda}$$
* **Posterior**:
  $$\lambda \mid y \sim \operatorname{Gamma}\left(\alpha + \sum_{i=1}^n y_i, \; \beta + n\right)$$
* **Posterior Mean**:
  $$\mathbb{E}[\lambda \mid y] = \frac{\alpha + \sum y_i}{\beta + n}$$


In [ ]:
# 📊 Interactive Beta-Binomial Updating Visualizer
theta_grid = np.linspace(0.001, 0.999, 400)
a_prior, b_prior = 2.0, 2.0
k_obs, n_obs = 24, 40

a_post = a_prior + k_obs
b_post = b_prior + n_obs - k_obs

prior_pdf = stats.beta.pdf(theta_grid, a_prior, b_prior)
lik_vals = stats.binom.pmf(k_obs, n_obs, theta_grid)
post_pdf = stats.beta.pdf(theta_grid, a_post, b_post)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_pdf, mode='lines', line=dict(color='#94a3b8', dash='dot', width=2), name=f'Prior Beta({a_prior},{b_prior})'))
fig.add_trace(go.Scatter(x=theta_grid, y=lik_vals * (post_pdf.max() / lik_vals.max()), mode='lines', line=dict(color='#d97706', dash='dash', width=2), name=f'Likelihood (Scaled)'))
fig.add_trace(go.Scatter(x=theta_grid, y=post_pdf, mode='lines', line=dict(color='#2563eb', width=3), name=f'Posterior Beta({a_post},{b_post})'))

fig.update_layout(
    title=f'Conjugate Update: Prior Beta({a_prior},{b_prior}) + Data ({k_obs}/{n_obs}) -> Posterior Beta({a_post},{b_post})',
    xaxis_title='Parameter θ',
    yaxis_title='Density',
    template='plotly_white',
    height=400
)
fig.show()


---
## 3. Task 1: Multiple-Choice Exam Success (Beta-Binomial)

### Scenario:
Two students take a multiple-choice exam consisting of **$N = 40$ questions**, where each question has **4 choices** (1 correct, 3 distractors).
* Random guessing corresponds to a baseline success rate of $\theta = 0.25$.
* We believe both students have studied and will perform better than chance, but we are uncertain about their exact mastery.
* **Student A** scores $k_A = 24 / 40$.
* **Student B** scores $k_B = 34 / 40$.

### Questions to Answer:
1. Define the parameter of interest $\theta$ and write the formal likelihood function.
2. Formulate a defensible Beta prior reflecting the belief that the student performs better than chance guessing ($> 0.25$). Explain your choice of $(\alpha, \beta)$.
3. Compute the exact posterior distributions for both Student A and Student B.
4. Calculate the posterior mean and $95\%$ equal-tailed credible interval for each student.
5. What is the posterior probability that Student A has a true mastery rate exceeding $60\%$ ($P(\theta_A > 0.60 \mid D)$)?


In [ ]:
# 🧪 Task 1: Complete Your Analysis Here
# Example Prior: Beta(alpha, beta) with mean > 0.25
# E.g. Prior mean = 0.50, prior effective sample size = 4 -> Beta(2, 2)
# E.g. Prior mean = 0.40, prior effective sample size = 10 -> Beta(4, 6)

alpha_prior = 4.0
beta_prior = 6.0

n_exam = 40
k_A = 24
k_B = 34

# 1. Update Student A
alpha_post_A = alpha_prior + k_A
beta_post_A = beta_prior + n_exam - k_A
dist_A = stats.beta(alpha_post_A, beta_post_A)

mean_A = dist_A.mean()
interval_A = dist_A.ppf([0.025, 0.975])
prob_A_gt_60 = 1.0 - dist_A.cdf(0.60)

# 2. Update Student B
alpha_post_B = alpha_prior + k_B
beta_post_B = beta_prior + n_exam - k_B
dist_B = stats.beta(alpha_post_B, beta_post_B)

mean_B = dist_B.mean()
interval_B = dist_B.ppf([0.025, 0.975])

print(f"Student A (24/40): Posterior Mean = {mean_A:.4f}, 95% ETI = [{interval_A[0]:.4f}, {interval_A[1]:.4f}], P(θ > 0.60) = {prob_A_gt_60*100:.2f}%")
print(f"Student B (34/40): Posterior Mean = {mean_B:.4f}, 95% ETI = [{interval_B[0]:.4f}, {interval_B[1]:.4f}]")


### 💡 Suggested Experiments for Task 1:
* **Experiment 1A (Guessing Threshold Cutoff)**: What is the posterior probability that Student A was merely guessing ($P(\theta_A \le 0.25 \mid D)$)?
* **Experiment 1B (Prior Sensitivity)**: Contrast your results under an uninformative prior $\operatorname{Beta}(1, 1)$ vs. a skeptical prior $\operatorname{Beta}(2.5, 7.5)$ centered exactly at the guessing rate $0.25$.


---
## 4. Task 2: Public Health Proportions (CDC BRFSS Survey)

### Scenario:
We analyze a representative sample from the [CDC Behavioral Risk Factor Surveillance System (BRFSS)](http://www.cdc.gov/brfss/).

### Tasks:
1. Load `data/brfss.csv` and inspect sample size and variable codings (`sex`, `exerany2`).
2. Construct a $95\%$ Bayesian credible interval for the proportion of the female population in the US.
3. Estimate the proportion of the population that performed any physical activity in the past 30 days (`exerany2 == 1`).
4. Evaluate how sensitive the estimated physical activity proportion is to using a flat $\operatorname{Beta}(1, 1)$ prior versus a weakly informative prior $\operatorname{Beta}(10, 10)$.


In [ ]:
# 🧪 Task 2: Load Data and Compute Proportions
df_brfss = pd.read_csv(LOCAL_PATH, index_col=0)
print(f"Loaded BRFSS dataset with {len(df_brfss):,} respondents.")
display(df_brfss.head(3))

# Check gender distribution (1 = Male, 2 = Female in standard CDC coding, or 'm'/'f')
print("\nGender counts:")
print(df_brfss['sex'].value_counts(dropna=False))

# 1. Female proportion inference
# Determine female indicator column
is_female = (df_brfss['sex'] == 2) | (df_brfss['sex'] == 'f') | (df_brfss['sex'] == 'Female')
k_female = is_female.sum()
n_gender = len(df_brfss)

# Beta(1,1) update
a_fem_post = 1 + k_female
b_fem_post = 1 + n_gender - k_female
dist_fem = stats.beta(a_fem_post, b_fem_post)

print(f"\nFemale Proportion: Sample = {k_female}/{n_gender} ({k_female/n_gender:.4f})")
print(f"Posterior Mean: {dist_fem.mean():.4f}")
print(f"95% Credible Interval: [{dist_fem.ppf(0.025):.4f}, {dist_fem.ppf(0.975):.4f}]")

# 2. Exercise proportion
# Column is 'exercise' ('Yes' / 'No')
is_exer = (df_brfss['exercise'] == 'Yes') | (df_brfss['exercise'] == 1)
k_exer = is_exer.sum()
n_exer = len(df_brfss['exercise'].dropna())

dist_exer = stats.beta(1 + k_exer, 1 + n_exer - k_exer)
print(f"\nPhysical Activity Proportion: Sample = {k_exer}/{n_exer} ({k_exer/n_exer:.4f})")
print(f"Posterior Mean: {dist_exer.mean():.4f}")
print(f"95% Credible Interval: [{dist_exer.ppf(0.025):.4f}, {dist_exer.ppf(0.975):.4f}]")


### 💡 Suggested Experiments for Task 2:
* **Subgroup Comparison**: Does physical activity differ by sex? Compute the posterior distributions for males and females separately and compare their credible intervals.


---
## 5. Task 3: Fruit & Vegetable Daily Intake (Gamma-Poisson)

### Scenario:
Public health authorities recommend consuming **4 to 5 portions** of fruits or vegetables per day.
In the BRFSS dataset, `fruit_per_day` and `vege_per_day` record daily servings. We sum them to obtain total daily intake $Y_i$.
Since daily servings are non-negative count integers, we model them using a **Poisson likelihood** with rate parameter $\lambda$ (expected servings per day).

### Tasks:
1. Formulate a Gamma prior $\lambda \sim \operatorname{Gamma}(\alpha, \beta)$ representing typical dietary intake.
2. Given observed total servings $\sum y_i$ across $n$ participants, compute the exact Gamma posterior.
3. Compute the posterior mean and $95\%$ equal-tailed credible interval for daily servings $\lambda$.
4. Is population intake in line with the recommended 4–5 portions per day? Calculate the posterior probability $\mathbb{P}(\lambda \ge 4.0 \mid D)$.


In [ ]:
# 🧪 Task 3: Gamma-Poisson Dietary Intake Model
# Total daily intake = fruit + vegetables
intake = (df_brfss['fruit_per_day'] + df_brfss['vege_per_day']).dropna()
sum_y = intake.sum()
n_diet = len(intake)

# Gamma prior: alpha=6, beta=2 (prior mean = 6/2 = 3.0 servings, weight = 2 prior respondents)
a_gamma_prior = 6.0
b_gamma_prior = 2.0

# Conjugate Gamma update
a_gamma_post = a_gamma_prior + sum_y
b_gamma_post = b_gamma_prior + n_diet
post_gamma_dist = stats.gamma(a=a_gamma_post, scale=1.0 / b_gamma_post)

mean_lambda = post_gamma_dist.mean()
interval_lambda = post_gamma_dist.ppf([0.025, 0.975])
prob_guideline = 1.0 - post_gamma_dist.cdf(4.0)

print(f"Intake Sample: n = {n_diet:,}, Mean = {intake.mean():.3f} servings/day")
print(f"Posterior Gamma: shape = {a_gamma_post:.1f}, rate = {b_gamma_post:.1f}")
print(f"Posterior Expected Servings/Day (λ): {mean_lambda:.4f}")
print(f"Probability Meeting Recommendation (λ ≥ 4.0): {prob_guideline*100:.4f}%")


### 💡 Suggested Experiments for Task 3:
* **Zero-Inflation Audit**: Dietary survey data often have excess zeros (people who report 0 fruit/vegetables). Compute what fraction of respondents report zero servings, and discuss whether a Zero-Inflated Poisson or Negative Binomial model would be a more authentic generative process.


---
## 6. Reproducibility Footer
* Course: Bayesian Analysis of Empirical Data (2026)
* Workbook: Session 4 Practice (Beta-Binomial & Gamma-Poisson Conjugates)
* Validated in Google Colab and local Python 3.12 environments
